In [1]:
!pip install chromadb FlagEmbedding sentence-transformers

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 163.9/163.9 kB 11.6 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 67.3/67.3 kB 4.3 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Preparing metadata (setup.py) ... done
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.7/21.7 MB 36.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 17.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 36.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.4/17.4 MB 91.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.5/72.5 kB 4.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 132.6/132.6 kB 6.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.4/66.4 kB 5.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━

In [3]:
import json
import os
import shutil
import sys
from pathlib import Path

# Detect if running in Colab to auto-install dependencies if this is a cell
try:
    import google.colab
    IN_COLAB = True
    print("✨ Detected Google Colab Environment")
    # Uncomment to auto-install in a script-like way,
    # but typically users run "!pip install..." in a cell before this.
    # os.system('pip install chromadb FlagEmbedding sentence-transformers')
except ImportError:
    IN_COLAB = False


✨ Detected Google Colab Environment


In [5]:
INPUT_FILENAME = "blade_views_enhanced.json"
OUTPUT_DB_FOLDER = "blade_views_chroma_db"
COLLECTION_NAME = "blade_views_knowledge"

# Paths
BASE_DIR = Path("/content") if IN_COLAB else Path.cwd()
INPUT_FILE = BASE_DIR / INPUT_FILENAME
CHROMA_DB_PATH = BASE_DIR / OUTPUT_DB_FOLDER

def main():
    # 0. Check Dependencies
    try:
        import chromadb
        from chromadb.config import Settings
        from FlagEmbedding import BGEM3FlagModel
    except ImportError as e:
        print("❌ Missing Dependencies.")
        print("Please run: !pip install chromadb FlagEmbedding sentence-transformers")
        return

    print("🚀 Starting Blade View Embedding (Colab T4 Edition)...")

    # 1. Load Chunks
    if not INPUT_FILE.exists():
        print(f"❌ Error: Input file not found at {INPUT_FILE}")
        print(f"👉 Please upload '{INPUT_FILENAME}' to the files area on the left.")
        return

    with open(INPUT_FILE, 'r', encoding='utf-8') as f:
        chunks = json.load(f)

    print(f"📂 Loaded {len(chunks)} chunks")

    # 2. Filter for Enhanced Chunks
    valid_chunks = [c for c in chunks if c.get('description_enhanced')]
    print(f"🔍 Found {len(valid_chunks)} enhanced chunks to embed")

    if not valid_chunks:
        print("⚠️  No enhanced chunks found. Make sure you uploaded the ENHANCED json file.")
        return

    # 3. Initialize ChromaDB
    print(f"🔧 Initializing ChromaDB at {CHROMA_DB_PATH}")

    # Ensure directory exists/reset path cleanup if needed (optional, relying on API below is safer)
    if CHROMA_DB_PATH.exists():
        # shutil.rmtree(CHROMA_DB_PATH) # Removing this to rely on API for safer re-runs
        pass

    client = chromadb.PersistentClient(
        path=str(CHROMA_DB_PATH),
        settings=Settings(anonymized_telemetry=False)
    )

    # Robustly handle existing collection
    try:
        client.delete_collection(name=COLLECTION_NAME)
        print(f"  🗑️  Deleted existing API collection: {COLLECTION_NAME}")
    except Exception as e:
        # Collection might not exist, which is fine
        pass

    collection = client.create_collection(
        name=COLLECTION_NAME,
        metadata={"embedding_model": "BAAI/bge-m3"}
    )

    # 4. Load Embedding Model (High Accuracy Mode)
    print("🤖 Loading Embedding Model (BAAI/bge-m3)...")
    print("  ⚡ Enable FP16: False (Prioritizing Accuracy)")
    print("  🐢 Tokenizer: Slow (use_fast=False)")

    # Disable FP16 to ensure maximum precision
    model = BGEM3FlagModel("BAAI/bge-m3", use_fp16=False)

    # Force Slow Tokenizer as requested
    from transformers import AutoTokenizer
    model.tokenizer = AutoTokenizer.from_pretrained("BAAI/bge-m3", use_fast=False)

    # 5. Generate Embeddings
    print("🔄 Generating Embeddings (Individual Processing)...")

    batch_size = 50
    total_batches = (len(valid_chunks) + batch_size - 1) // batch_size

    for i in range(0, len(valid_chunks), batch_size):
        batch = valid_chunks[i:i+batch_size]

        batch_ids = [c['chunk_id'] for c in batch]
        batch_codes = [c['content'] for c in batch]

        # Prepare metadata first
        batch_metadatas = []
        for c in batch:
            meta = c.get('metadata', {}).copy()
            meta['file_name'] = c.get('file_name', '')
            meta['section'] = c.get('section_name', '')
            meta['description'] = c.get('description', '')

            clean_meta = {}
            for k, v in meta.items():
                if isinstance(v, (str, int, float, bool)):
                    clean_meta[k] = v
                else:
                    clean_meta[k] = str(v)
            batch_metadatas.append(clean_meta)

        print(f"  ⚡ Embedding batch {i//batch_size + 1}/{total_batches}...", end="", flush=True)

        # Embed individually to ensure proper tokenization and no batch-padding artifacts
        batch_embeddings = []
        for desc in [c['description'] for c in batch]:
            # Max length 8192 to capture full context if needed
            emb = model.encode(desc, max_length=8192)['dense_vecs']
            batch_embeddings.append(emb.tolist())

        collection.add(
            ids=batch_ids,
            embeddings=batch_embeddings,
            documents=batch_codes,
            metadatas=batch_metadatas
        )
        print(" Done ✅")

    print(f"\n🎉 Successfully embedded {len(valid_chunks)} chunks!")

    # 6. Zip for Download
    print("📦 Zipping database for download...")
    zip_path = shutil.make_archive(OUTPUT_DB_FOLDER, 'zip', CHROMA_DB_PATH)
    print(f"✅ Created: {zip_path}")
    print("\n👉 Dowload 'blade_views_chroma_db.zip' from the files sidebar!")

if __name__ == "__main__":
    main()

🚀 Starting Blade View Embedding (Colab T4 Edition)...
📂 Loaded 179 chunks
🔍 Found 177 enhanced chunks to embed
🔧 Initializing ChromaDB at /content/blade_views_chroma_db
🤖 Loading Embedding Model (BAAI/bge-m3)...
  ⚡ Enable FP16: False (Prioritizing Accuracy)
  🐢 Tokenizer: Slow (use_fast=False)


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/444 [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.1M [00:00<?, ?B/s]

...

onnx/tokenizer.json:   0%|          | 0.00/17.1M [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/54.0 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/2.27G [00:00<?, ?B/s]

sparse_linear.pt:   0%|          | 0.00/3.52k [00:00<?, ?B/s]

🔄 Generating Embeddings (Individual Processing)...
  ⚡ Embedding batch 1/4... Done ✅
  ⚡ Embedding batch 2/4... Done ✅
  ⚡ Embedding batch 3/4... Done ✅
  ⚡ Embedding batch 4/4... Done ✅

🎉 Successfully embedded 177 chunks!
📦 Zipping database for download...
✅ Created: /content/blade_views_chroma_db.zip

👉 Dowload 'blade_views_chroma_db.zip' from the files sidebar!
